# `source` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'source'
feature_metadata = {'order': 35, 'name': 'source', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain as granular source representation', 'finding': 'Ten fully covered levels map deterministically upward and preserve differences hidden by broader types.', 'decision': 'Retain source and compare source_type as a lower-cardinality alternative.', 'risk': 'Keeping every source hierarchy level adds deterministic redundancy.', 'sentinel_tokens': ['unknown'], 'related': [{'feature': 'source_type', 'reason': 'This is the deterministic intermediate parent.'}, {'feature': 'source_class', 'reason': 'This is the deterministic broad parent.'}, {'feature': 'basin', 'reason': 'Hydrological basin shapes available water sources.'}, {'feature': 'extraction_type', 'reason': 'Extraction mechanism depends on source.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for source.


## Supported target evidence


In [2]:
sentinel_tokens = ['unknown']
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
source,,,,,
spring,17021,True,62.23,7.50,30.27
shallow well,16824,True,49.48,5.69,44.83
machine dbh,11075,True,48.96,4.43,46.61
river,9612,True,56.86,12.70,30.44
rainwater harvesting,2295,True,60.39,13.68,25.93
hand dtw,874,True,56.86,1.95,41.19
lake,765,True,21.18,1.57,77.25
dam,656,True,38.57,3.66,57.77
other,212,True,59.43,0.47,40.09


status_group,rows,non functional (%)
source,,
lake,765,77.25
dam,656,57.77
machine dbh,11075,46.61
shallow well,16824,44.83
hand dtw,874,41.19
other,212,40.09
river,9612,30.44
spring,17021,30.27
rainwater harvesting,2295,25.93


## Observation

Ten fully covered levels map deterministically upward and preserve differences hidden by broader types.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Retain source and compare source_type as a lower-cardinality alternative.

**Risk to carry forward:** Keeping every source hierarchy level adds deterministic redundancy.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
source,candidate,retain as granular source representation,Ten fully covered levels map deterministically...,Retain source and compare source_type as a low...,Keeping every source hierarchy level adds dete...
